# From Linear Models to a Neural Network

Notebook for **Chapter 1** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch01.html

Tip: change a number, predict the output, then run the cell again.

## 1.1 The Perceptron: One Artificial Neuron

In [ ]:
import numpy as np
X = np.array([[0,0],[0,1],[1,0],[1,1]])

def train(y, epochs=10):
    w = np.zeros(2); b = 0.0
    for ep in range(epochs):
        err = 0
        for x, t in zip(X, y):
            p = 1 if w @ x + b > 0 else 0
            d = t - p
            if d:
                w = w + d * x; b += d; err += 1
        if err == 0:
            return w, b, ep
    return w, b, None

for name, y in [("AND",[0,0,0,1]), ("OR",[0,1,1,1]), ("XOR",[0,1,1,0])]:
    print(name, train(np.array(y)))

AND (array([2., 1.]), np.float64(-2.0), 5)
OR (array([1., 1.]), np.float64(0.0), 3)
XOR (array([-1.,  0.]), np.float64(1.0), None)


## 1.2 From One Neuron to a Multi-Layer Perceptron (MLP)

In [ ]:
import numpy as np
X = np.array([[0,0],[0,1],[1,0],[1,1]])
W1 = np.array([[1,1],[1,1]]); b1 = np.array([-0.5,-1.5])
W2 = np.array([1,-1]);        b2 = -0.5
step = lambda z: (z > 0).astype(int)
H = step(X @ W1 + b1)
print(H)
print(step(H @ W2 + b2))

[[0 0]
 [1 0]
 [1 0]
 [1 1]]
[0 1 1 0]


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
A = rng.normal(size=(2,3)); B = rng.normal(size=(4,2)); x = rng.normal(size=3)
print(np.allclose(B @ (A @ x), (B @ A) @ x), (B @ A).shape)

True (4, 3)


## 1.3 Activation Functions and Why Each One Exists

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
z = np.array([-3,-1,0,1,3.])
sig  = lambda z: 1/(1+np.exp(-z))
relu = lambda z: np.maximum(0,z)
gelu = lambda z: 0.5*z*(1+np.tanh(np.sqrt(2/np.pi)*(z+0.044715*z**3)))
silu = lambda z: z*sig(z)
print("z   ",z);print("relu",relu(z));print("sig ",sig(z))
print("tanh",np.tanh(z));print("gelu",gelu(z));print("silu",silu(z))
print("sig'",sig(z)*(1-sig(z)))

z    [-3. -1.  0.  1.  3.]
relu [0. 0. 0. 1. 3.]
sig  [0.0474 0.2689 0.5    0.7311 0.9526]
tanh [-0.9951 -0.7616  0.      0.7616  0.9951]
gelu [-0.0036 -0.1588  0.      0.8412  2.9964]
silu [-0.1423 -0.2689  0.      0.7311  2.8577]
sig' [0.0452 0.1966 0.25   0.1966 0.0452]


In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
s = np.array([2.0,1.0,0.1]); e = np.exp(s); print(e, e/e.sum())
s2 = np.array([1002.,1001,1000]); m = np.exp(s2 - s2.max()); print(m/m.sum())
print(np.exp(s2))

[7.3891 2.7183 1.1052] [0.659  0.2424 0.0986]
[0.6652 0.2447 0.09  ]
[inf inf inf]


## 1.4 Start From Zero: Words, a Neuron on Paper, and Batches

Sections 1.1 to 1.3 move fast. This section slows down. If any word in the first three sections felt like jargon, read this one first, then go back. Everything here uses the smallest numbers possible. You will build the one idea the whole chapter rests on: **a neuron multiplies, adds, and bends**.

In [ ]:
import numpy as np
x = np.array([2.0, 3.0, 1.0])
w = np.array([0.5, -0.4, 1.2])
b = -0.3
z = w @ x + b
print("z      =", round(z, 4))
print("sigmoid=", round(1 / (1 + np.exp(-z)), 4))
print("step   =", int(z > 0))
print("relu   =", round(max(0.0, z), 4))
print("tanh   =", round(np.tanh(z), 4))

z      = 0.7
sigmoid= 0.6682
step   = 1
relu   = 0.7
tanh   = 0.6044


In [ ]:
import numpy as np
x = np.array([2.0, 3.0, 1.0]); w = np.array([0.5, -0.4, 1.2])
sigmoid = lambda z: 1 / (1 + np.exp(-z))
for b in [-3, -1, 0, 1, 3]:
    z = round(w @ x + b, 6) + 0.0
    print(f"b={b:+d}  z={z:+.1f}  sigmoid={sigmoid(z):.4f}")

b=-3  z=-2.0  sigmoid=0.1192
b=-1  z=+0.0  sigmoid=0.5000
b=+0  z=+1.0  sigmoid=0.7311
b=+1  z=+2.0  sigmoid=0.8808
b=+3  z=+4.0  sigmoid=0.9820


In [ ]:
import numpy as np
X = np.array([[2, 3, 1], [1, 10, 0], [5, 1, 1], [0, 4, 0]], dtype=float)
w = np.array([0.5, -0.4, 1.2]); b = -0.3
z = X @ w + b
p = 1 / (1 + np.exp(-z))
print("z    ", np.round(z, 4))
print("p    ", np.round(p, 4))
print("late?", (p > 0.5).astype(int))
loop = np.array([w @ row + b for row in X])
print("loop equals matrix:", np.allclose(loop, z))

z     [ 0.7 -3.8  3.  -1.9]
p     [0.6682 0.0219 0.9526 0.1301]
late? [1 0 1 0]
loop equals matrix: True


## 1.5 The Perceptron Up Close: the Boundary, the Distance and the Training Trace

Section 1.1 said "a perceptron draws a straight line". Now we make that sentence exact, so you can draw the line, measure how far each point is from it, design weights by hand, and predict how training behaves.

In [ ]:
import numpy as np
w = np.array([2.0, 1.0]); b = -2.0
corners = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
z = corners @ w + b
dist = z / np.linalg.norm(w)
print("norm of w :", round(np.linalg.norm(w), 4))
print("z         :", z)
print("distance  :", np.round(dist, 4))
print("outputs   :", (z > 0).astype(int))
scaled = corners @ (10 * w) + 10 * b
print("same outputs after scaling by 10:", np.array_equal(scaled > 0, z > 0))

norm of w : 2.2361
z         : [-2. -1.  0.  1.]
distance  : [-0.8944 -0.4472  0.      0.4472]
outputs   : [0 0 0 1]
same outputs after scaling by 10: True


In [ ]:
import numpy as np
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
gates = {
    "AND":      ((1, 1), -1.5),
    "OR":       ((1, 1), -0.5),
    "NAND":     ((-1, -1), 1.5),
    "NOR":      ((-1, -1), 0.5),
    "x1 !x2":   ((1, -1), -0.5),
}
print("inputs     ", [tuple(r) for r in X])
for name, (w, b) in gates.items():
    out = ((X @ np.array(w) + b) > 0).astype(int)
    print(f"{name:10s}", out)

inputs      [(np.int64(0), np.int64(0)), (np.int64(0), np.int64(1)), (np.int64(1), np.int64(0)), (np.int64(1), np.int64(1))]
AND        [0 0 0 1]
OR         [0 1 1 1]
NAND       [1 1 1 0]
NOR        [1 0 0 0]
x1 !x2     [0 0 1 0]


In [ ]:
import numpy as np
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]); y = np.array([0, 1, 1, 1])
w = np.zeros(2); b = 0.0
for epoch in range(1, 5):
    updates = 0
    for x, t in zip(X, y):
        p = 1 if w @ x + b > 0 else 0
        d = t - p
        if d != 0:
            w = w + d * x; b = b + d; updates += 1
            print(f"epoch {epoch} x={tuple(int(v) for v in x)} target={t} p={p} -> w={tuple(int(v) for v in w)} b={int(b)}")
    if updates == 0:
        print(f"epoch {epoch}: clean pass, stop"); break

epoch 1 x=(0, 1) target=1 p=0 -> w=(0, 1) b=1
epoch 2 x=(0, 0) target=0 p=1 -> w=(0, 1) b=0
epoch 2 x=(1, 0) target=1 p=0 -> w=(1, 1) b=1
epoch 3 x=(0, 0) target=0 p=1 -> w=(1, 1) b=0
epoch 4: clean pass, stop


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
P = rng.uniform(-3, 3, size=(200, 2))
P = P[np.abs(P @ np.array([1.0, 1.0]) / np.sqrt(2)) > 0.5]
y = (P @ np.array([1.0, 1.0]) > 0).astype(int)
A = np.hstack([P, np.ones((len(P), 1))])          # append constant 1
w_star = np.array([1.0, 1.0, 0.0]) / np.sqrt(2)   # unit-length true separator
gamma = np.min(np.abs(A @ w_star))
R = np.max(np.linalg.norm(A, axis=1))
w = np.zeros(3); updates = 0
for epoch in range(1000):
    mistakes = 0
    for a, t in zip(A, y):
        d = t - (1 if w @ a > 0 else 0)
        if d != 0:
            w = w + d * a; updates += 1; mistakes += 1
    if mistakes == 0:
        break
print("points kept   :", len(P))
print("gamma, R      :", round(gamma, 4), round(R, 4))
print("bound (R/g)^2 :", round((R / gamma) ** 2, 1))
print("real updates  :", updates, "| clean pass at epoch index", epoch)

points kept   : 155
gamma, R      : 0.502 4.2689
bound (R/g)^2 : 72.3
real updates  : 8 | clean pass at epoch index 2


## 1.6 Hidden Layers in Practice: Building Functions, Tracing Shapes, Counting Memory

Section 1.2 showed that a hidden layer rescues XOR. Here you learn to *build* the weights for a target by hand, to see how many bends a network can make, to trace tensor shapes through a batch, and to turn a parameter count into bytes. These four skills answer most exam, interview and debugging questions about plain networks.

In [ ]:
import numpy as np
relu = lambda v: np.maximum(0, v)
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]], dtype=float)
W1 = np.array([[1.0, 1.0], [1.0, 1.0]])    # both hidden neurons read x1 + x2
b1 = np.array([0.0, -1.0])
W2 = np.array([1.0, -2.0]); b2 = 0.0
H = relu(X @ W1 + b1)
y = H @ W2 + b2
print("hidden:"); print(H)
print("output:", y)

hidden:
[[0. 0.]
 [1. 0.]
 [1. 0.]
 [2. 1.]]
output: [0. 1. 1. 0.]


In [ ]:
import numpy as np
from itertools import product
relu = lambda v: np.maximum(0, v)
X = np.array(list(product([0, 1], repeat=3)), dtype=float)   # all 8 inputs
W1 = np.ones((3, 3)); b1 = np.array([0.0, -1.0, -2.0])
W2 = np.array([1.0, -2.0, 2.0])
y = relu(X @ W1 + b1) @ W2
parity = X.sum(axis=1) % 2
print(np.c_[X, y, parity].astype(int))
print("all correct:", np.array_equal(y, parity))

[[0 0 0 0 0]
 [0 0 1 1 1]
 [0 1 0 1 1]
 [0 1 1 0 0]
 [1 0 0 1 1]
 [1 0 1 0 0]
 [1 1 0 0 0]
 [1 1 1 1 1]]
all correct: True


In [ ]:
import numpy as np
relu = lambda v: np.maximum(0, v)
f = lambda x: x ** 2
xs = np.linspace(-2, 2, 4001)
for k in [2, 4, 8]:
    knots = np.linspace(-2, 2, k + 1)
    slopes = np.diff(f(knots)) / np.diff(knots)
    out_w = np.concatenate([[slopes[0]], np.diff(slopes)])   # slope changes
    yhat = f(-2) + sum(out_w[i] * relu(xs - knots[i]) for i in range(k))
    print(f"k={k}: output weights {np.round(out_w, 3)}  max error {np.abs(yhat - f(xs)).max():.4f}")

k=2: output weights [-2.  4.]  max error 1.0000
k=4: output weights [-3.  2.  2.  2.]  max error 0.2500
k=8: output weights [-3.5  1.   1.   1.   1.   1.   1.   1. ]  max error 0.0625


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
sizes = [20, 64, 32, 1]
Ws = [rng.normal(size=(i, o)) * 0.1 for i, o in zip(sizes[:-1], sizes[1:])]
bs = [np.zeros(o) for o in sizes[1:]]
h = rng.normal(size=(32, 20))
print("input", h.shape)
for k, (W, b) in enumerate(zip(Ws, bs), 1):
    z = h @ W + b
    h = z if k == len(Ws) else np.maximum(0, z)
    print(f"layer {k}: W {W.shape} b {b.shape} gives output {h.shape}")
print("parameters:", sum(W.size + b.size for W, b in zip(Ws, bs)))
bad = rng.normal(size=(64, 20))
try:
    rng.normal(size=(32, 20)) @ bad
except ValueError as e:
    print("error:", str(e))

input (32, 20)
layer 1: W (20, 64) b (64,) gives output (32, 64)
layer 2: W (64, 32) b (32,) gives output (32, 32)
layer 3: W (32, 1) b (1,) gives output (32, 1)
parameters: 3457
error: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 64 is different from 20)


In [ ]:
sizes = [784, 128, 64, 10]
layer = [i * o + o for i, o in zip(sizes[:-1], sizes[1:])]
total = sum(layer)
print("per layer:", layer, "total:", total)
print("float32 bytes:", total * 4, "MiB:", round(total * 4 / 2**20, 3))
print("Adam training (4 copies) MiB:", round(total * 16 / 2**20, 3))
print("logistic regression:", 784 * 10 + 10, "ratio:", round(total / 7850, 1))
print("7B fp16 GB:", 7e9 * 2 / 1e9, " 7B mixed-precision Adam GB:", 7e9 * 16 / 1e9)

per layer: [100480, 8256, 650] total: 109386
float32 bytes: 437544 MiB: 0.417
Adam training (4 copies) MiB: 1.669
logistic regression: 7850 ratio: 13.9
7B fp16 GB: 14.0  7B mixed-precision Adam GB: 112.0


## 1.7 Activations Under the Microscope: Slopes, Dead Neurons, Temperature and Stable Losses

Section 1.3 told you what each activation looks like. Training only cares about two things: the *value* in the forward pass and the *slope* in the backward pass. This section measures both, shows you the two classic failures (vanishing gradients and dead neurons), and shows how to compute softmax and cross-entropy without overflowing. Chapter 2 uses these slopes to train.

In [ ]:
import numpy as np
sig = lambda z: 1 / (1 + np.exp(-z))
silu = lambda z: z * sig(z)
z, eps = 2.0, 1e-6
numeric = lambda f: (f(z + eps) - f(z - eps)) / (2 * eps)
rows = [
    ("sigmoid", sig(z) * (1 - sig(z)),               numeric(sig)),
    ("tanh",    1 - np.tanh(z) ** 2,                 numeric(np.tanh)),
    ("silu",    sig(z) + z * sig(z) * (1 - sig(z)),  numeric(silu)),
]
for name, formula, num in rows:
    print(f"{name:8s} formula {formula:.4f}  numeric {num:.4f}")

sigmoid  formula 0.1050  numeric 0.1050
tanh     formula 0.0707  numeric 0.0707
silu     formula 1.0908  numeric 1.0908


In [ ]:
import numpy as np
def backward_norms(kind, L=10, n=50, seed=0):
    rng = np.random.default_rng(seed)
    scale = np.sqrt(2 / n) if kind == "relu" else np.sqrt(1 / n)
    Ws = [rng.normal(size=(n, n)) * scale for _ in range(L)]
    h = rng.normal(size=n); acts, zs = [h], []
    for W in Ws:                                   # forward
        z = W @ h; zs.append(z)
        h = np.maximum(0, z) if kind == "relu" else 1 / (1 + np.exp(-z))
        acts.append(h)
    g = np.ones(n); norms = []
    for i in reversed(range(L)):                   # backward
        slope = (zs[i] > 0).astype(float) if kind == "relu" else acts[i + 1] * (1 - acts[i + 1])
        g = g * slope
        norms.append(np.linalg.norm(g))
        g = Ws[i].T @ g
    return norms[::-1]                             # index 0 = layer 1

for kind in ["sigmoid", "relu"]:
    n = backward_norms(kind)
    print(f"{kind:8s} layer1 {n[0]:.2e}  layer5 {n[4]:.2e}  layer10 {n[9]:.2e}  last/first {n[9] / n[0]:.3g}")

sigmoid  layer1 3.23e-06  layer5 1.52e-03  layer10 1.71e+00  last/first 5.28e+05
relu     layer1 1.27e+01  layer5 1.09e+01  layer10 5.10e+00  last/first 0.401


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
X = rng.normal(size=(10000, 2))
w = np.array([1.0, 1.0])
for b in [0, -1, -3, -10]:
    z = X @ w + b
    active = (z > 0).mean()
    leaky_slope = np.where(z > 0, 1.0, 0.01).mean()
    print(f"bias {b:+3d}: active {active:.4f}  avg ReLU slope {active:.4f}  avg leaky slope {leaky_slope:.4f}")

bias  +0: active 0.5041  avg ReLU slope 0.5041  avg leaky slope 0.5091
bias  -1: active 0.2412  avg ReLU slope 0.2412  avg leaky slope 0.2488
bias  -3: active 0.0158  avg ReLU slope 0.0158  avg leaky slope 0.0256
bias -10: active 0.0000  avg ReLU slope 0.0000  avg leaky slope 0.0100


In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
s = np.array([2.0, 1.0, 0.1])
for T in [0.25, 1, 2, 5]:
    e = np.exp((s - s.max()) / T)
    print(f"T={T:<5}", e / e.sum())
s2 = np.array([1002.0, 1001.0, 1000.0])
m = s2.max()
log_softmax = (s2 - m) - np.log(np.exp(s2 - m).sum())
print("log-softmax:", log_softmax)
print("probabilities back:", np.exp(log_softmax))

T=0.25  [0.9815 0.018  0.0005]
T=1     [0.659  0.2424 0.0986]
T=2     [0.5017 0.3043 0.194 ]
T=5     [0.3996 0.3272 0.2733]
log-softmax: [-0.4076 -1.4076 -2.4076]
probabilities back: [0.6652 0.2447 0.09  ]


In [ ]:
import numpy as np
z, y = 40.0, 0
with np.errstate(all="ignore"):
    p = 1 / (1 + np.exp(-z))
    naive = -(y * np.log(p) + (1 - y) * np.log(1 - p))
stable = max(z, 0) - z * y + np.log1p(np.exp(-abs(z)))
print("p      =", p)
print("naive  =", naive)
print("stable =", stable)

p      = 1.0
naive  = inf
stable = 40.0


## 1.8 How to Solve Any "From Linear Models to a Neural Network" Problem

Every question about this chapter, whether in a quiz, an interview or a debugging session, is one of eight patterns. This section gives you a decision flow to recognise the pattern in under a minute, a solved example of each pattern, a checklist to verify any answer, and a debugging table for when you are stuck. Work the examples with a pencil before you read the solution. The skill you are building is not memory; it is knowing what to do first.

In [ ]:
def count(sizes, bias=True):
    return sum(i * o + (o if bias else 0) for i, o in zip(sizes[:-1], sizes[1:]))

sizes = [100, 50, 50, 10]
a = count(sizes)
b = a - 50
print("with biases:", a)
print("first layer without bias:", b)
print("float32 MB:", a * 4 / 1e6)
print("weights only:", count(sizes, bias=False), " biases:", a - count(sizes, bias=False))

with biases: 8110
first layer without bias: 8060
float32 MB: 0.03244
weights only: 8000  biases: 110


In [ ]:
import numpy as np
rng = np.random.default_rng(0)
images = rng.normal(size=(32, 1, 28, 28))
flat = images.reshape(32, -1)
print("flattened:", flat.shape)
bad = rng.normal(size=(28, 256))
try:
    flat @ bad
except ValueError as e:
    print("bug reproduced:", type(e).__name__, "(784 vs", bad.shape[0], ")")
W1 = rng.normal(size=(784, 256)) * 0.05; W2 = rng.normal(size=(256, 10)) * 0.05
out = np.maximum(0, flat @ W1) @ W2
print("fixed output:", out.shape)

flattened: (32, 784)
bug reproduced: ValueError (784 vs 28 )
fixed output: (32, 10)


In [ ]:
import numpy as np
x = np.array([1.0, -1.0, 2.0])
W1 = np.array([[1.0, 0.0, -1.0], [0.5, 0.5, 0.5]]); b1 = np.array([0.5, 0.0])
W2 = np.array([2.0, 3.0]); b2 = -1.0
z1 = W1 @ x + b1
h = np.maximum(0, z1)
z2 = W2 @ h + b2
p = 1 / (1 + np.exp(-z2))
print("z1 =", z1, " h =", h)
print("z2 =", z2, " p =", round(p, 4))
print("loss if y=1:", round(-np.log(p), 4), " loss if y=0:", round(-np.log(1 - p), 4))

z1 = [-0.5  1. ]  h = [0. 1.]
z2 = 2.0  p = 0.8808
loss if y=1: 0.1269  loss if y=0: 2.1269


In [ ]:
import numpy as np
from itertools import product
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
weights = np.arange(-2, 2.01, 1.0); biases = np.arange(-2.5, 2.51, 0.5)
found = set()
for w1, w2, b in product(weights, weights, biases):
    found.add(tuple(((X @ np.array([w1, w2]) + b) > 0).astype(int)))
print("rules one neuron can compute:", len(found), "of 16")
missing = [t for t in product([0, 1], repeat=4) if t not in found]
print("cannot compute:", missing)

step = lambda v: (v > 0).astype(int)
xs = np.array([0.0, 1.0, 1.5, 2.0, 2.5, 3.0])
out = step(step(xs - 1) - step(xs - 2) - 0.5)
print("x  :", xs)
print("out:", out)

rules one neuron can compute: 14 of 16
cannot compute: [(0, 1, 1, 0), (1, 0, 0, 1)]
x  : [0.  1.  1.5 2.  2.5 3. ]
out: [0 0 1 1 0 0]


In [ ]:
import numpy as np
pts = np.array([[2, 1], [-1, -2], [2, -1], [3, 0]], dtype=float); t = np.array([1, 0, 0, 1])
w = np.zeros(2); b = 0.0
for epoch in range(1, 10):
    updates = 0
    for x, target in zip(pts, t):
        z = w @ x + b
        p = 1 if z > 0 else 0
        if p != target:
            w = w + (target - p) * x; b += target - p; updates += 1
            print(f"epoch {epoch}: x={x.astype(int).tolist()} z={z:+.0f} target={target} -> w={w.astype(int).tolist()} b={b:.0f}")
    if updates == 0:
        print(f"epoch {epoch} clean. final w={w.astype(int).tolist()} b={b:.0f}"); break

epoch 1: x=[2, 1] z=+0 target=1 -> w=[2, 1] b=1
epoch 1: x=[2, -1] z=+4 target=0 -> w=[0, 2] b=0
epoch 1: x=[3, 0] z=+0 target=1 -> w=[3, 2] b=1
epoch 2: x=[2, -1] z=+5 target=0 -> w=[1, 3] b=0
epoch 3 clean. final w=[1, 3] b=0


In [ ]:
import numpy as np
s = np.array([2.0, 1.0, 0.1])
softmax = np.exp(s - s.max()) / np.exp(s - s.max()).sum()
sigmoids = 1 / (1 + np.exp(-s))
print("softmax :", np.round(softmax, 4), "sum", round(softmax.sum(), 4))
print("sigmoids:", np.round(sigmoids, 4), "sum", round(sigmoids.sum(), 4))

softmax : [0.659  0.2424 0.0986] sum 1.0
sigmoids: [0.8808 0.7311 0.525 ] sum 2.1368


In [ ]:
import numpy as np
z = np.array([90.0, 0.0, -90.0], dtype=np.float32)
target = 2
with np.errstate(all="ignore"):
    e = np.exp(z)
    p = e / e.sum()
    naive = -np.log(p[target])
m = z.max()
stable = np.log(np.exp(z - m).sum()) + m - z[target]
print("exp(z)   :", e)
print("naive p  :", p)
print("naive loss :", naive)
print("stable loss:", stable)

exp(z)   : [        inf 1.00000e+00 8.19401e-40]
naive p  : [nan  0.  0.]
naive loss : inf
stable loss: 180.0


In [ ]:
import numpy as np
def ratio(act, scale, L=8, n=50, seed=0):
    rng = np.random.default_rng(seed)
    Ws = [rng.normal(size=(n, n)) * scale for _ in range(L)]
    f = {"sigmoid": lambda z: 1 / (1 + np.exp(-z)), "tanh": np.tanh, "relu": lambda z: np.maximum(0, z)}[act]
    h = rng.normal(size=n); H, Z = [h], []
    for W in Ws:
        Z.append(W @ H[-1]); H.append(f(Z[-1]))
    slope = {"sigmoid": lambda i: H[i + 1] * (1 - H[i + 1]),
             "tanh": lambda i: 1 - H[i + 1] ** 2,
             "relu": lambda i: (Z[i] > 0) * 1.0}[act]
    g = np.ones(n); norms = []
    for i in reversed(range(L)):
        g = g * slope(i); norms.append(np.linalg.norm(g)); g = Ws[i].T @ g
    return norms[-1] / norms[0]            # layer 1 over layer L

cases = [("sigmoid", np.sqrt(1 / 50)), ("tanh", np.sqrt(1 / 50)), ("relu", 0.01),
         ("relu", np.sqrt(2 / 50)), ("relu", 0.5)]
for act, scale in cases:
    r = ratio(act, scale)
    verdict = "vanishing" if r < 0.1 else "exploding" if r > 10 else "healthy"
    print(f"{act:8s} scale {scale:.3f}  layer1/layer8 = {r:.2e}  {verdict}")

sigmoid  scale 0.141  layer1/layer8 = 3.17e-05  vanishing
tanh     scale 0.141  layer1/layer8 = 2.70e-01  healthy
relu     scale 0.010  layer1/layer8 = 2.70e-09  vanishing
relu     scale 0.200  layer1/layer8 = 3.45e+00  healthy
relu     scale 0.500  layer1/layer8 = 2.11e+03  exploding


## 1.9 Going Deeper: Expert Notes, Edge Cases and What Interviewers Probe

This section is for when the basics feel solid. Each part answers a question that a strong interviewer or a senior reviewer would ask, and each has a number you can verify. Skip it on a first reading and come back after Chapter 2.

In [ ]:
import numpy as np
rng = np.random.default_rng(1)
W = rng.normal(size=(8, 2)); b = rng.normal(size=8)
g = np.linspace(-6, 6, 600)
GX, GY = np.meshgrid(g, g)
pts = np.stack([GX.ravel(), GY.ravel()], axis=1)
patterns = pts @ W.T + b > 0
print("distinct on/off patterns:", len({tuple(r) for r in patterns}), "| bound:", 1 + 8 + 28)

relu = lambda v: np.maximum(0, v)
tent = lambda x: 2 * relu(x) - 4 * relu(x - 0.5)
xs = np.linspace(0, 1, 100001)
y = xs.copy()
for k in range(1, 5):
    y = tent(y)
    slopes = np.round(np.diff(y) / np.diff(xs), 3)
    pieces = int((np.abs(np.diff(slopes)) > 1e-6).sum()) + 1
    print(f"{k} tent layers ({2 * k} neurons): {pieces} straight pieces")

distinct on/off patterns: 36 | bound: 37
1 tent layers (2 neurons): 2 straight pieces
2 tent layers (4 neurons): 4 straight pieces
3 tent layers (6 neurons): 8 straight pieces
4 tent layers (8 neurons): 16 straight pieces


In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def softmax(v):
    e = np.exp(v - v.max()); return e / e.sum()
z = np.array([2.0, 1.0, 0.1])
s = softmax(z)
J = np.diag(s) - np.outer(s, s)
eps = 1e-6
Jn = np.stack([(softmax(z + eps * np.eye(3)[j]) - softmax(z - eps * np.eye(3)[j])) / (2 * eps) for j in range(3)], axis=1)
print(J)
print("max difference to numeric:", f"{np.abs(J - Jn).max():.1e}")
print("column sums:", np.round(J.sum(axis=0), 10))
print("shift invariant:", np.allclose(softmax(z), softmax(z + 1000)))

[[ 0.2247 -0.1598 -0.065 ]
 [-0.1598  0.1837 -0.0239]
 [-0.065  -0.0239  0.0889]]
max difference to numeric: 7.9e-11
column sums: [0. 0. 0.]
shift invariant: True


In [ ]:
import numpy as np
from scipy.special import erf
z = np.linspace(-5, 5, 10001)
exact = 0.5 * z * (1 + erf(z / np.sqrt(2)))
approx = 0.5 * z * (1 + np.tanh(np.sqrt(2 / np.pi) * (z + 0.044715 * z ** 3)))
print("at z=1  exact", round(float(0.5 * (1 + erf(1 / np.sqrt(2)))), 4), " approx", round(float(approx[np.argmin(np.abs(z - 1))]), 4))
print("largest difference on [-5, 5]:", f"{np.abs(exact - approx).max():.6f}")
print("minimum of exact GELU:", round(float(exact.min()), 4), "at z =", round(float(z[exact.argmin()]), 2))

at z=1  exact 0.8413  approx 0.8412
largest difference on [-5, 5]: 0.000473
minimum of exact GELU: -0.17 at z = -0.75


In [ ]:
import numpy as np
with np.errstate(all="ignore"):
    print("max float16        :", np.finfo(np.float16).max)
    print("exp(11), exp(12)   :", np.exp(np.float16(11)), np.exp(np.float16(12)))
    print("2048 + 1 in fp16   :", np.float16(2048) + np.float16(1))
    print("2048 + 1 in fp32   :", np.float32(2048) + np.float32(1))
    s16 = np.float16(1) / (np.float16(1) + np.exp(np.float16(-12)))
    print("sigmoid(12) in fp16:", s16, "| in fp64:", round(float(1 / (1 + np.exp(-12.0))), 7))

max float16        : 6.55e+04
exp(11), exp(12)   : 5.987e+04 inf
2048 + 1 in fp16   : 2.048e+03
2048 + 1 in fp32   : 2049.0
sigmoid(12) in fp16: 1.0 | in fp64: 0.9999939


In [ ]:
i = o = 4096
params = i * o + o
weight_bytes = params * 2
flops = 2 * i * o
print("parameters   :", params)
print("weight MB    :", round(weight_bytes / 1e6, 1))
print("FLOPs/example:", flops)
for batch in [1, 128]:
    print(f"batch {batch:3d}: FLOPs per byte of weights = {batch * flops / weight_bytes:.1f}")

parameters   : 16781312
weight MB    : 33.6
FLOPs/example: 33554432
batch   1: FLOPs per byte of weights = 1.0
batch 128: FLOPs per byte of weights = 128.0


In [ ]:
import numpy as np
sig = lambda z: 1 / (1 + np.exp(-z))
x = np.array([1.0, 2.0]); y = 1.0
W1 = np.full((2, 2), 0.5); b1 = np.zeros(2)
W2 = np.full(2, 0.5); b2 = 0.0
h = sig(W1 @ x + b1)
p = sig(W2 @ h + b2)
dz2 = p - y
dz1 = dz2 * W2 * h * (1 - h)
dW1 = np.outer(dz1, x)
print("h:", np.round(h, 4), " p:", round(p, 4))
print("gradient rows for the two hidden neurons:")
print(np.round(dW1, 5))
print("identical rows:", np.allclose(dW1[0], dW1[1]))

h: [0.8176 0.8176]  p: 0.6937
gradient rows for the two hidden neurons:
[[-0.02284 -0.04568]
 [-0.02284 -0.04568]]
identical rows: True


## 1.10 Mixed Drill, Self-Test and Where to Go Next

Real problems do not announce their section. This drill mixes everything in the chapter. Cover the answers, work each item on paper in under two minutes, then check. After the drill comes a scored self-test that tells you exactly what to reread.

## Chapter recap

## Practice set

**1.** (easy) How many parameters in an MLP with layers 4 → 8 → 3? **Answer:** `4·8+8 = 40` and `8·3+3 = 27`, total 67.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch01.html